## Multiagent System

PS: BA gathers requirement from the client. Reverify (Human in the loop) the requirement with the client after it said its requirement.

If client agrees- passes to Manager.

If client desagress- update the requirement and reverify with client.

Manager takes the requirement and give this requirement to team lead. The team lead understand the requirement and creates a 1 sprint task. and assign it to the devlopers.

The developer starts working, once done , the code is given to QA.

If QA is satisfy, QA Passes the completion  msg to Team lead.

Team lead gives the update to Manager.

Manager updates the BA

BA updates the Client.

Client approves if he is satisfied.

If yes, Congrulations, Else the flow goes on again!



In [1]:
from typing import Annotated, Literal, Optional
from pathlib import Path
import sys
from IPython.display import display, Image
from dotenv import load_dotenv
from langchain.messages import SystemMessage
from langchain_openai import ChatOpenAI, data
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph, add_messages
import sqlite3
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage  , HumanMessage,trim_messages ,RemoveMessage
from langgraph.checkpoint.sqlite import SqliteSaver
import json
from pydantic import BaseModel, Field
import os

In [2]:
llm=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3
)

In [7]:
class GlobalState(BaseModel):
    requirement:str =Field(description="Requirement gathered by BA")
    client_query:str=Field(description="Clients  raw requirement")
    BA_MSG:str=Field(description="Remark of Buisness Analyst")
    Manager_MSG:str=Field(description="Remark of Manager")
    Teamlead_MSG:str=Field(description="Remark of Team Lead")
    QA_MSG:str=Field(description="Remark of Quality Assurance staff")
    Developer_MSG:str=Field(description="Remark of Developers")
    current_stage: Literal['BA','MANAGER','TEAM_LEAD','QA','DEVELOPER'] = "BA"
    next_stage: Literal['BA','MANAGER','TEAM_LEAD','QA','DEVELOPER']
    is_completed : bool = False
    is_requirement_clear: bool = False

In [ ]:
# Making Agents Node

class BA_response(BaseModel):
    requirement:str= Field(description="Evaluated and refined client requirement.")
    BA_MSG:str=Field(description="Additional msg to the team/ Manager.")

def BA(State:GlobalState):
    ba_prompt = ChatPromptTemplate.from_template('''
You are a Senior Business Analyst. Your task is to evaluate and refine the following client requirement.

### Objective
Review the provided input to determine if it contains sufficient detail for technical implementation. 

### Instructions

1. If the requirement is vague, ambiguous, or lacks critical context:
   - Identify the gaps in information.
   - Generate a maximum of three (3) highly specific, professional clarification questions for the client.
   - Focus questions on scope, user roles, or expected outcomes.

2. If the requirement is clear and actionable:
   - Structure the information into a professional Business Requirements Document (BRD) format optimized for management review.
   - Include sections for: Executive Summary, Target Users, Core Functional Requirements, and Success Criteria.
   - Use clear, professional, and structured language suitable for stakeholders and project managers.
                                                 - You can also give additional message to MANAGER.

### Client Input
{client_requirement}
''')


    llm_with_structured_output=llm.with_structured_output(BA_response)
    res=ba_prompt | llm_with_structured_output
    content=res.invoke({"client_requirement" : State.client_query })

    return {
        State.requirement:content.requirement,
        State.BA_MSG: content.BA_MSG,
        State.is_requirement_clear:True,
        State.current_stage:"BA",
        State.next_stage:"MANAGER"
    }

class Manger_response(BaseModel):
    Manager_MSG : str = Field(description='Additional message which you will give to Team lead along with the requirements.')
    AllowedTech = Literal[
    'Python', 'FastAPI', 'AWS', 'React js', 'HTML', 
    'CSS', 'Redis', 'Vector DB (Pinecone)', 'MCP', 'MySQL'
]
    
    

def MANAGER (State:GlobalState):
    manager_prompt = ChatPromptTemplate.from_template(
    '''You are an IT Project Manager/Architect. Your role is to analyze project `requirements` from the Business Analyst and select the optimal technical tools for project completion. 
    You are also responsible for assigning work tasks to the Team Lead and evaluating the final deliverables submitted by the Team Lead.
    You must review the `BA_msg` to incorporate feedback or special remarks from the Business Analyst.

    Your team is highly proficient in the following core technology stack:
    ['Python', 'FastAPI', 'AWS', 'React.js', 'HTML', 'CSS', 'Redis', 'Vector DB (Pinecone)', 'MCP', 'MySQL']
    You may define the architecture using this stack or closely related technologies.

    ### REQUIREMENTS
    {requirements}

    ### BUSINESS ANALYST REMARKS (BA_msg)
    {BA_MSG}
    '''
    )

    llm_with_structured_output=llm.with_structured_output(Manger_response)
    chain= manager_prompt | llm_with_structured_output
    content=chain.invoke({
        "requirements":State.requirement,
        "BA_MSG":State.BA_MSG
    })

    return {
        State.requirement:content.requirement,
        State.Manager_MSG: content.Manager_MSG,
        State.is_requirement_clear:True,
        State.current_stage:"MANAGER",
        State.next_stage:"TEAM_LEAD"
    }

PydanticUserError: A non-annotated attribute was detected: `AllowedTech = typing.Literal['Python', 'FastAPI', 'AWS', 'React js', 'HTML', 'CSS', 'Redis', 'Vector DB (Pinecone)', 'MCP', 'MySQL']`. All model fields require a type annotation; if `AllowedTech` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.13/u/model-field-missing-annotation